In [0]:
# ============================================================
# NOTEBOOK 1: BRONZE LAYER — FIXED
# Team AryaBit | HackBricks 2026
# ============================================================

import logging
import re
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("AryaBit.Bronze")

# ── CONFIG ──────────────────────────────────────────────────
CATALOG        = "workspace"
SCHEMA         = "default"
VOLUME_PATH    = f"/Volumes/{CATALOG}/{SCHEMA}/hackbricks_data"
SOURCE_FILE    = f"{VOLUME_PATH}/students_dropout_academic_success.csv"
BRONZE_TABLE   = f"{CATALOG}.{SCHEMA}.bronze_student_dropout"

# ── STEP 1: SANITIZE COLUMN NAMES ───────────────────────────
def sanitize_column_names(df):
    """
    Replace all invalid characters in column names with underscores.
    Delta Lake doesn't allow: spaces , ; { } ( ) \n \t =
    """
    def clean(col_name: str) -> str:
        # Replace invalid chars with underscore
        cleaned = re.sub(r"[ ,;{}\(\)\n\t=\/\'\-]+", "_", col_name)
        # Remove trailing underscores
        cleaned = cleaned.strip("_")
        # Lowercase everything
        cleaned = cleaned.lower()
        return cleaned

    new_columns = [clean(c) for c in df.columns]
    
    # Log the mapping so we know what changed
    print("\nCOLUMN NAME MAPPING:")
    for old, new in zip(df.columns, new_columns):
        if old != new:
            print(f"  '{old}' → '{new}'")
    
    return df.toDF(*new_columns)

# ── STEP 2: READ RAW CSV ────────────────────────────────────
def read_raw_data(source_path: str):
    logger.info(f"Reading raw data from: {source_path}")
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("sep", ",")
        .csv(source_path)
    )
    logger.info(f"Raw record count: {df.count()}")
    return df

# ── STEP 3: ADD BRONZE METADATA ─────────────────────────────
def add_bronze_metadata(df):
    return (
        df
        .withColumn("_ingestion_timestamp", current_timestamp())
        .withColumn("_source_file", lit(SOURCE_FILE))
        .withColumn("_layer", lit("bronze"))
    )

# ── STEP 4: WRITE TO DELTA TABLE ────────────────────────────
def write_bronze_table(df, table_name: str):
    logger.info(f"Writing Bronze Delta table: {table_name}")
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )
    logger.info(f"Bronze table written: {table_name}")

# ── STEP 5: DOCUMENT SCHEMA ─────────────────────────────────
def document_schema(table_name: str):
    print("\n" + "="*60)
    print("BRONZE TABLE SCHEMA DOCUMENTATION")
    print("="*60)
    spark.sql(f"DESCRIBE TABLE {table_name}").show(50, truncate=False)
    
    print("\nSAMPLE RECORDS (5):")
    spark.sql(f"SELECT * FROM {table_name} LIMIT 5").show(truncate=False)
    
    print("\nRECORD COUNT:")
    count = spark.sql(
        f"SELECT COUNT(*) as total FROM {table_name}"
    ).collect()[0][0]
    print(f"Total records ingested: {count}")
    
    print("\nTARGET DISTRIBUTION:")
    spark.sql(f"""
        SELECT 
            target,
            COUNT(*) as count,
            ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
        FROM {table_name}
        GROUP BY target
        ORDER BY target
    """).show()

# ── MAIN EXECUTION ───────────────────────────────────────────
def run_bronze_pipeline():
    logger.info("Starting Bronze Layer Pipeline — AryaBit")
    try:
        # Read raw
        raw_df = read_raw_data(SOURCE_FILE)
        
        # Sanitize column names — CRITICAL STEP
        clean_df = sanitize_column_names(raw_df)
        
        # Add metadata
        bronze_df = add_bronze_metadata(clean_df)
        
        # Write Delta
        write_bronze_table(bronze_df, BRONZE_TABLE)
        
        # Document
        document_schema(BRONZE_TABLE)
        
        logger.info("Bronze Layer COMPLETE ✅")
        return True
        
    except Exception as e:
        logger.error(f"Bronze Pipeline FAILED: {str(e)}")
        raise e

# ── RUN ──────────────────────────────────────────────────────
run_bronze_pipeline()